# 04 — Cold-start integrity and evaluation

## Goal

Review cold-drug and cold-target manifest diagnostics, then compare matched random and cold-start experiment records. This notebook does not recreate or reshuffle partitions.


## Setup


In [ ]:
from pathlib import Path
from importlib import import_module
import json
import subprocess
import sys


def find_project_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "config.py").is_file() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the project checkout.")


def require_api(module_name: str, attribute: str):
    module = import_module(module_name)
    if not hasattr(module, attribute):
        raise RuntimeError(
            f"Required v2 API {module_name}.{attribute} is unavailable. "
            "Update v2-refactor and rerun the tests before enabling this cell."
        )
    return getattr(module, attribute)


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

BRANCH = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=PROJECT_ROOT, text=True
).strip()
if BRANCH != "v2-refactor":
    raise RuntimeError(f"Expected v2-refactor, found {BRANCH!r}. v1-final is read-only.")

print(f"Project: {PROJECT_ROOT}")
print(f"Branch:  {BRANCH}")


In [ ]:
from config import EXPERIMENT_LOG_PATH, SPLITS_DIR

RUN_INTEGRITY_REVIEW = False
RUN_GENERALISATION_SUMMARY = False
print("Split metadata:", SPLITS_DIR / "metadata")
print("Experiment log:", EXPERIMENT_LOG_PATH)


## Steps


In [ ]:
if RUN_INTEGRITY_REVIEW:
    metadata_files = sorted((SPLITS_DIR / "manifests").rglob("*.json"))
    metadata_files += sorted((SPLITS_DIR / "metadata").rglob("*.json"))
    if not metadata_files:
        raise FileNotFoundError("No split metadata JSON files were found.")
    for path in metadata_files:
        payload = json.loads(path.read_text(encoding="utf-8"))
        print(path.relative_to(PROJECT_ROOT), json.dumps(payload, indent=2))
else:
    print("Integrity review skipped: RUN_INTEGRITY_REVIEW is False.")


In [ ]:
if RUN_GENERALISATION_SUMMARY:
    import pandas as pd

    records = pd.read_csv(EXPERIMENT_LOG_PATH)
    cold = records[records["split"].isin(["cold_drug", "cold_target"])]
    columns = [
        "dataset", "model", "split", "seed", "best_validation_mse",
        "test_rmse", "concordance_index", "pearson_correlation",
    ]
    print(cold[columns].sort_values(
        ["dataset", "model", "seed", "split"]
    ).to_string(index=False, max_rows=100))
else:
    print("Generalisation summary skipped: RUN_GENERALISATION_SUMMARY is False.")


## Checks

Cold-drug must have no source drug-ID overlap and cold-target must have no source target-ID overlap across all three partitions. Pair overlap must always be zero. Also report identical-SMILES and identical-sequence overlaps as separate diagnostics.

## Next steps

Resolve any integrity failure before training. Structure/sequence-disjoint supplementary splits require an explicit methodology decision and must not silently replace v1 ID-based splits.
